## **Setup & Load Data**

In [ ]:
# install.packages("FNN")
# install.packages("ggrepel")
library(FNN)
library(lubridate)
library(tidyverse)
library(readr)
library(stringr)
library(bigrquery)
library(parallel)
library(ggrepel)

In [ ]:
EXPORT_BUCKET = "gs://plm-pers-hba1c-thresh-wb-silky-pepper-6055"
BILLING = "wb-silky-pepper-6055"
DATA_MOUNT = "/home/jupyter/workspace/raw/vwb-aou-datasets-controlled/v8"

In [ ]:
# Read query data directly from Cloud Storage into memory
read_bq_export_from_workspace_bucket <- function(export_path) {
  col_types <- NULL
  bind_rows(
    map(system2('gsutil', args = c('ls', export_path), stdout = TRUE, stderr = TRUE),
        function(csv) {
          message(str_glue('Loading {csv}.'))
          chunk <- read_csv(pipe(str_glue('gsutil cat {csv}')), col_types = col_types, show_col_types = FALSE)
          if (is.null(col_types)) {
            col_types <- spec(chunk)
          }
          chunk
        }))
}

In [ ]:
#TODO: need to revisit this - the auto-generated paths are only different by date, not cohort!
person_path <- "gs://plm-pers-hba1c-thresh-wb-silky-pepper-6055/bq_exports/person/20260509/person/person_*.csv"
person_df <- read_bq_export_from_workspace_bucket(person_path)

measurementOccurrence_path <- "gs://plm-pers-hba1c-thresh-wb-silky-pepper-6055/bq_exports/measurementOccurrence/20260510/measurementOccurrence/measurementOccurrence_*.csv"
measure_df <- read_bq_export_from_workspace_bucket(measurementOccurrence_path)

In [ ]:
# Read ancestry predictions
#TODO: why does this workspace have echo_v4_r2 prefix?
ancestry_path = paste0(DATA_MOUNT, "/wgs/short_read/snpindel/aux/ancestry/echo_v4_r2.ancestry_preds.tsv")
ancestry_raw <- read_tsv(
  ancestry_path,
  show_col_types = FALSE,
  progress = FALSE)

## **Get Mean Shift for Every Individual**

Mean shift is calculated by using the shift in maxmimum density in the whole cohort compared to the matched cohort.

In [ ]:
# Global Density Peak
dim(data_anal)
full_d <- density(data_anal$tsh, na.rm = TRUE)
full_d_peak <- full_d$x[which.max(full_d$y)]

summary(data_anal$tsh)
print(full_d_peak)

In [ ]:
# For every person
    # Calculate their cohort's maximum A1c density
    # Calculate the shifted thresholds
get_shift <- function(id, cohort_ids) {
    cohort_ids <- unlist(cohort_ids, use.names = F)
    
    cohort <- data_anal[data_anal$person_id %in% cohort_ids, ]
    #TODO: could add average number measures per person in each cohort

    cohort_d <- density(cohort$tsh)  # checks on NA done & range already set
    cohort_d_peak <- cohort_d$x[which.max(cohort_d$y)]
    #TODO: read more about whether different density bandwidth should be used?

    shift <- full_d_peak - cohort_d_peak
    return(shift)
}

In [ ]:
#TO NOTE: these two rows must use the same ID list based on order
neighbor_list <- lapply(ids_anal, function(id) knn_cohort_t[, id])

In [ ]:
# detectCores()  # 4
# TO NOTE: this takes 20+ minutes as currently written

shift_values <- mclapply(seq_along(ids_anal), function(i) {
  id <- ids_anal[i]
  cohort_ids <- neighbor_list[[i]]
  get_shift(id, cohort_ids)
}, mc.cores = 4)

df <- data.frame(id = ids_anal, shift = unlist(shift_values))

In [ ]:
# Save file
write_tsv(df, "tsh_anal_shift.txt")
system(paste0("gsutil cp tsh_anal_shift.txt ", EXPORT_BUCKET, "/"))

In [ ]:
# Reload file
system(paste0("gsutil cp ", EXPORT_BUCKET, "/tsh_anal_shift.txt ./"))
df <- read.table("tsh_anal_shift.txt", header = TRUE)
dim(df)
head(df)